In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: Qiskit Function od Q-CTRL Fire Opal
*Viz [referenci API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (prostřednictvím IBM Quantum Platform API) Plan. Jsou ve stavu preview verze a mohou se měnit.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Přehled
S Fire Opal Optimization Solver můžeš řešit optimalizační problémy v utility-scale měřítku na kvantovém hardwaru bez potřeby kvantové odbornosti. Jednoduše zadej definici problému na vysoké úrovni a Solver se postará o zbytek. Celý pracovní postup je noise-aware a využívá pod pokličkou [Fire Opal Performance Management](/guides/q-ctrl-performance-management). Solver konzistentně poskytuje přesná řešení klasicky náročných problémů, a to i v plném rozsahu zařízení na největších IBM&reg; QPU.

Solver je flexibilní a lze jej použít k řešení kombinatorických optimalizačních problémů definovaných jako účelové funkce nebo libovolné grafy. Problémy nemusí být mapovány na topologii zařízení. Řešitelné jsou jak neomezené, tak omezené problémy za předpokladu, že omezení lze formulovat jako penalizační členy. Příklady obsažené v tomto průvodci ukazují, jak vyřešit neomezený a omezený optimalizační problém v utility-scale měřítku pomocí různých typů vstupů Solveru. První příklad zahrnuje problém max-cut definovaný na grafu s 156 uzly a 3-regulárním uspořádáním, zatímco druhý příklad řeší problém Minimum Vertex Cover s 50 uzly definovaný pomocí účelové funkce.

Chceš-li získat přístup k Optimization Solver, [kontaktuj Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Popis funkce
Solver plně optimalizuje a automatizuje celý algoritmus – od potlačování chyb na úrovni hardwaru až po efektivní mapování problému a uzavřenou smyčku klasické optimalizace. V zákulisí pipeline Solveru snižuje chyby v každé fázi, což umožňuje zvýšený výkon potřebný pro smysluplné škálování. Základní pracovní postup je inspirován algoritmem Quantum Approximate Optimization Algorithm (QAOA), který je hybridním kvantově-klasickým algoritmem. Podrobný přehled celého pracovního postupu Optimization Solver najdeš v [publikovaném rukopisu](https://arxiv.org/abs/2406.01743).

![Vizualizace pracovního postupu Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Jak vyřešit obecný problém pomocí Optimization Solver:
1. Definuj svůj problém jako účelovou funkci, graf nebo `SparsePauliOp` spin chain.
2. Připoj se k funkci prostřednictvím katalogu Qiskit Functions.
3. Spusť problém pomocí Solveru a načti výsledky.
### Přijímané formáty problémů
- Polynomiální vyjádření účelové funkce. Ideálně vytvořené v Pythonu s existujícím objektem SymPy Poly a formátované do řetězce pomocí [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Grafová reprezentace konkrétního typu problému. Graf by měl být vytvořen pomocí knihovny networkx v Pythonu a poté převeden na řetězec pomocí funkce networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Reprezentace spin chain konkrétního problému. Spin chain by měl být reprezentován jako objekt `SparsePauliOp`; více podrobností najdeš v [dokumentaci](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp).

> **Note:** Pokud chceš použít Backend, který tato funkce aktuálně nepodporuje, [kontaktuj Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) a přidej jeho podporu.
## Benchmarky
[Publikované výsledky benchmarkingu](https://arxiv.org/abs/2406.01743) ukazují, že Solver úspěšně řeší problémy s více než 120 qubity a dokonce překonává dříve publikované výsledky na kvantových žíhacích a trapped-ion zařízeních. Následující metriky benchmarku poskytují hrubý přehled o přesnosti a škálování typů problémů na základě několika příkladů. Skutečné metriky se mohou lišit v závislosti na různých vlastnostech problému, jako je počet členů v účelové funkci (hustota) a jejich lokalita, počet proměnných a polynomiální řád.

Uvedený „Počet qubitů" není pevným omezením, ale představuje přibližné prahové hodnoty, kde lze očekávat mimořádně konzistentní přesnost řešení. Větší velikosti problémů byly úspěšně vyřešeny a testování za těmito hranicemi je podporováno.

Libovolná konektivita qubitů je podporována napříč všemi typy problémů.

| Typ problému    | Počet qubitů | Příklad | Přesnost | Celkový čas (s) | Využití Runtime (s) | Počet iterací
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Řídce propojené kvadratické problémy  | 156 | 3-regulární max-cut| 100%     | 1764     | 293          | 16 |
| Binární optimalizace vyššího řádu | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Hustě propojené kvadratické problémy | 50 | Plně propojený max-cut| 100%      |  1758    | 268  | 12 |
| Omezený problém s penalizačními členy | 50 | Vážené Minimum Vertex Cover s hustotou hran 8 % | 100%      | 1074     | 215 | 10 |
## Začínáme
Nejprve se ověř pomocí svého [IBM Quantum API klíče](http://quantum.cloud.ibm.com/). Poté vyber Qiskit Function takto. (Tento úryvek předpokládá, že jsi již [uložil svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do svého lokálního prostředí.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Definuj problém
Problém Max-Cut můžeš spustit definováním grafového problému a zadáním `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [5]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [6]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. Spusť problém
Při použití vstupní metody na základě grafu zadej typ problému.

In [7]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [8]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [9]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [10]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

ba5fbdb8-9e71-49bd-8320-79dcdb46de6a
QUEUED


### 3. Načti výsledek
Načti optimální hodnotu řezu ze slovníku výsledků.

> **Note:** Mapování proměnných na bitstring se mohlo změnit. Výstupní slovník obsahuje podslovník `variables_to_bitstring_index_map`, který pomáhá ověřit pořadí.

In [11]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [12]:
# %pip install numpy networkx sympy

Přesnost výsledku můžeš ověřit klasickým řešením problému pomocí open-source solverů jako [PuLP](https://coin-or.github.io/pulp/), pokud graf není hustě propojený. Problémy s vysokou hustotou mohou pro ověření řešení vyžadovat pokročilé klasické solvery.
## Příklad: Omezená optimalizace
Předchozí příklad max-cut je běžný kvadratický neomezený binární optimalizační problém. Optimization Solver od Q-CTRL lze použít pro různé typy problémů, včetně omezené optimalizace. Libovolné typy problémů můžeš řešit zadáním definice problému reprezentované jako polynom, kde jsou omezení modelována jako penalizační členy.

Následující příklad ukazuje, jak sestavit účelovou funkci pro omezený optimalizační problém – [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Kromě balíčků `qiskit-ibm-catalog` a `qiskit` budeš k tomuto příkladu také potřebovat: `numpy`, `networkx` a `sympy`. Tyto balíčky můžeš nainstalovat odkomentováním následující buňky, pokud spouštíš tento příklad v notebooku pomocí IPython kernelu.

In [13]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Definuj problém
Definuj náhodný problém MVC vygenerováním grafu s náhodně váženými uzly.

In [14]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

![Výstup předchozí buňky kódu](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Standardní optimalizační model pro vážené MVC lze formulovat takto. Nejprve musí být přidána penalizace pro každý případ, kdy hrana není spojena s vrcholem v podmnožině. Proto nechť $n_i = 1$, pokud je vrchol $i$ v pokrytí (tj. v podmnožině), a $n_i = 0$ v opačném případě. Zadruhé, cílem je minimalizovat celkový počet vrcholů v podmnožině, což lze vyjádřit následující funkcí:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [15]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

Nyní každá hrana v grafu musí obsahovat alespoň jeden koncový bod z pokrytí, což lze vyjádřit nerovností:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Každý případ, kdy hrana není spojena s vrcholem pokrytí, musí být penalizován. To lze vyjádřit v účelové funkci přidáním penalizačního členu ve tvaru $P(1-n_i-n_j+n_i n_j)$, kde $P$ je kladná penalizační konstanta. Neomezená alternativa k omezené nerovnosti pro vážené MVC je tedy:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [16]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. Spusť problém

In [17]:
print(mvc_job.status())

QUEUED


Zkontroluj [stav](/guides/functions#check-job-status) své pracovní zátěže Qiskit Function nebo načti [výsledky](/guides/functions#retrieve-results) takto:

In [18]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>